# Per-sample cost across profiling modalities

Rough per-sample costs for genomics, transcriptomics, proteomics and high-content imaging (Cell Painting).

**Caveat:** no single source prices all four modalities side by side. Figures come from separate papers and core-facility
price lists, with different bases (reagents only vs. full service), depths and years. Treat them as orders of magnitude.
Upper bounds quoted as "under X" or "exceeds X" are plotted at X and flagged in the table.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

GBP_TO_USD = 1.27  # assumed rate for the Edinburgh (GBP) price; adjust as needed

## Data

`basis`: `reagents` = reagents only, `sequencing` = data generation only, `service` = full core-facility service. `bound` marks a quoted limit (`<` or `>`) rather than an exact price.

In [ ]:
rows = [
    # modality, method, low, high, basis, bound, source, url
    (
        "Genomics",
        "WGS 30x, sequencing only (NovaSeq X)",
        200,
        200,
        "sequencing",
        "",
        "STAT News",
        "https://www.statnews.com/?p=868564",
    ),
    (
        "Genomics",
        "WGS, MGI DNBSEQ-T20x2 at scale",
        100,
        100,
        "sequencing",
        "<",
        "Labiotech",
        "https://www.labiotech.eu/in-depth/genome-sequencing-review-projects/",
    ),
    (
        "Genomics",
        "WGS 30x, full service",
        494,
        510,
        "service",
        "",
        "OHRI / TGen / Labiotech",
        "https://ohri.ca/fr/node/117131",
    ),
    (
        "Genomics",
        "Whole exome 100x, full service",
        204,
        204,
        "service",
        "",
        "OHRI / TGen",
        "https://tgen.org/service-centers/collaborative-sequencing-center/sequencing-costs",
    ),
    (
        "Transcriptomics",
        "DRUG-seq (384/1536-well)",
        2,
        4,
        "sequencing",
        "",
        "Ye et al., Nat Commun 2018",
        "https://pmc.ncbi.nlm.nih.gov/articles/6192987",
    ),
    (
        "Transcriptomics",
        "3'Pool-seq",
        15,
        15,
        "service",
        "<",
        "BMC Genomics 2020",
        "https://pmc.ncbi.nlm.nih.gov/articles/PMC6971924",
    ),
    (
        "Transcriptomics",
        "QuantSeq 3' kit (prep only)",
        25,
        25,
        "reagents",
        ">",
        "BMC Genomics 2020",
        "https://pmc.ncbi.nlm.nih.gov/articles/PMC6971924",
    ),
    (
        "Transcriptomics",
        "Bulk RNA-seq, full length",
        160,
        160,
        "service",
        ">",
        "BMC Genomics 2020",
        "https://pmc.ncbi.nlm.nih.gov/articles/PMC6971924",
    ),
    (
        "Transcriptomics",
        "Single-cell 10x 3' GEX (1 sample)",
        2400,
        2400,
        "service",
        "",
        "Weill Cornell Epigenomics Core",
        "https://epicore.med.cornell.edu/pricelist.php",
    ),
    (
        "Proteomics (MS)",
        "Bulk MS, Edinburgh (GBP 50-100)",
        50 * GBP_TO_USD,
        100 * GBP_TO_USD,
        "service",
        "",
        "Edinburgh IGC",
        "https://institute-genetics-cancer.ed.ac.uk/mass-spectrometry/pricing-and-access",
    ),
    (
        "Proteomics (MS)",
        "Bulk MS, Washington State",
        166,
        340,
        "service",
        "",
        "WSU TIMPL",
        "https://labs.wsu.edu/tipl/?p=77",
    ),
    (
        "Proteomics (MS)",
        "Phosphoproteomics, Pitt",
        450,
        550,
        "service",
        "",
        "Pitt Mass Spectrometry Core",
        "https://www.sharedfacilities.pitt.edu/msc/pricing",
    ),
    (
        "Proteomics (affinity)",
        "Olink Target96",
        95,
        108,
        "service",
        "",
        "Univ. of Houston CIBC",
        "https://cibc.bme.uh.edu/?p=1808",
    ),
    (
        "Proteomics (affinity)",
        "Olink Reveal (1,000-plex)",
        170,
        195,
        "service",
        "",
        "Univ. of Houston CIBC",
        "https://cibc.bme.uh.edu/?p=1808",
    ),
    (
        "Proteomics (affinity)",
        "SomaScan 7K",
        650,
        750,
        "service",
        "",
        "Univ. of Houston CIBC",
        "https://cibc.bme.uh.edu/?p=1808",
    ),
    (
        "Proteomics (affinity)",
        "SomaScan 11K",
        730,
        840,
        "service",
        "",
        "Univ. of Houston CIBC",
        "https://cibc.bme.uh.edu/?p=1808",
    ),
    (
        "Imaging",
        "Cell Painting (per well, stain reagents only)",
        0.26,
        0.26,
        "reagents",
        "",
        "Tromans-Coia et al., Cytometry A 2023",
        "https://pmc.ncbi.nlm.nih.gov/articles/PMC10841730",
    ),
]
df = pd.DataFrame(
    rows,
    columns=[
        "modality",
        "method",
        "low_usd",
        "high_usd",
        "basis",
        "bound",
        "source",
        "url",
    ],
)
df["mid_usd"] = (df.low_usd * df.high_usd) ** 0.5
df.drop(columns="url").round(2)

## Per-method cost range (log scale)

In [ ]:
colors = {
    "Imaging": "#009E73",
    "Transcriptomics": "#0072B2",
    "Proteomics (MS)": "#D55E00",
    "Proteomics (affinity)": "#CC79A7",
    "Genomics": "#E69F00",
}
order = [
    "Imaging",
    "Transcriptomics",
    "Proteomics (MS)",
    "Proteomics (affinity)",
    "Genomics",
]
d = (
    df.assign(m=pd.Categorical(df.modality, order, ordered=True))
    .sort_values(["m", "mid_usd"], ascending=[False, False])
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 8))
for y, r in d.iterrows():
    c = colors[r.modality]
    if r.high_usd > r.low_usd:
        ax.hlines(y, r.low_usd, r.high_usd, color=c, lw=6, alpha=0.85)
    ax.plot(
        r.mid_usd if r.high_usd > r.low_usd else r.low_usd,
        y,
        marker="o",
        ms=9,
        color=c,
        markerfacecolor="white" if r.basis == "reagents" else c,
        mew=2,
    )
    label = f"${r.low_usd:,.2f}" if r.low_usd < 1 else f"${r.low_usd:,.0f}"
    if r.high_usd > r.low_usd:
        label += f"-{r.high_usd:,.0f}"
    if r.bound:
        label = r.bound + label
    ax.annotate(
        label,
        (r.high_usd, y),
        xytext=(8, 0),
        textcoords="offset points",
        va="center",
        fontsize=8,
    )

ax.set_yticks(range(len(d)))
ax.set_yticklabels(d.method, fontsize=9)
ax.set_xscale("log")
ax.set_xlim(0.1, 8000)
ax.set_xlabel("Cost per sample (USD, log scale)")
ax.set_title("Per-sample cost by modality", loc="left", fontweight="bold")
ax.grid(axis="x", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

handles = [Line2D([0], [0], color=colors[m], lw=6, label=m) for m in reversed(order)]
handles += [
    Line2D(
        [0],
        [0],
        marker="o",
        color="grey",
        ls="",
        markerfacecolor="white",
        mew=2,
        label="reagents only",
    ),
    Line2D(
        [0], [0], marker="o", color="grey", ls="", label="sequencing / full service"
    ),
]
ax.legend(handles=handles, loc="upper right", frameon=False, fontsize=9)
fig.text(
    0.01,
    -0.01,
    "Sources are heterogeneous (reagents vs. full service, different years). '<' / '>' = quoted bound. "
    f"GBP converted at {GBP_TO_USD}.",
    fontsize=8,
    color="grey",
)
fig.tight_layout()
out_dir = Path("../figures")
out_dir.mkdir(exist_ok=True)
fig.savefig(
    out_dir / "cost_per_sample.png", dpi=600, bbox_inches="tight", facecolor="white"
)
plt.show()

## Modality-level summary

Cheapest to most expensive span across the methods above.

In [ ]:
s = (
    df.groupby("modality")
    .agg(min_usd=("low_usd", "min"), max_usd=("high_usd", "max"))
    .loc[order]
)

fig, ax = plt.subplots(figsize=(8, 3.5))
for y, (m, r) in enumerate(s.iterrows()):
    ax.hlines(y, r.min_usd, r.max_usd, color=colors[m], lw=10)
    ax.annotate(
        f"${r.min_usd:,.2f} - ${r.max_usd:,.0f}",
        (r.max_usd, y),
        xytext=(8, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
    )
ax.set_yticks(range(len(s)))
ax.set_yticklabels(s.index)
ax.set_xscale("log")
ax.set_xlim(0.1, 20000)
ax.set_xlabel("Cost per sample (USD, log scale)")
ax.set_title("Span of per-sample cost by modality", loc="left", fontweight="bold")
ax.grid(axis="x", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## Sources

- [STAT News: Illumina NovaSeq X launch](https://www.statnews.com/?p=868564)
- [Labiotech: Genome sequencing review](https://www.labiotech.eu/in-depth/genome-sequencing-review-projects/)
- [OHRI NGS service pricing](https://ohri.ca/fr/node/117131)
- [TGen sequencing costs](https://tgen.org/service-centers/collaborative-sequencing-center/sequencing-costs)
- [3'Pool-seq, BMC Genomics 2020](https://pmc.ncbi.nlm.nih.gov/articles/PMC6971924)
- [DRUG-seq, Nature Communications 2018](https://pmc.ncbi.nlm.nih.gov/articles/6192987)
- [Weill Cornell Epigenomics Core price list](https://epicore.med.cornell.edu/pricelist.php)
- [Edinburgh IGC mass spectrometry pricing](https://institute-genetics-cancer.ed.ac.uk/mass-spectrometry/pricing-and-access)
- [WSU TIMPL proteomics fees](https://labs.wsu.edu/tipl/?p=77)
- [Pitt Mass Spectrometry Core pricing](https://www.sharedfacilities.pitt.edu/msc/pricing)
- [University of Houston CIBC affinity proteomics pricing](https://cibc.bme.uh.edu/?p=1808)
- [Tromans-Coia et al., Cytometry A 2023: Cell Painting across imaging systems](https://pmc.ncbi.nlm.nih.gov/articles/PMC10841730)
- [Way et al., Cell Systems 2022 (preprint)](https://www.biorxiv.org/content/10.1101/2021.10.21.465335.full.pdf)
- [Way et al., Cell Systems 2022 (PMC)](https://pmc.ncbi.nlm.nih.gov/articles/PMC10246468)

Cell Painting's $0.26/well is stain reagents only; plates, cells, microscope time, storage and analysis are excluded.